# Step 7 · Design reward rules

Turn 'what a good answer looks like' into scoring rules — then try to fool them.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
SAVE = False   # set True to write new versions into your lab (they appear in the dashboard)

In [ ]:
rules = rewards.DEFAULT_RULES
pd.DataFrame(rules)[["name", "type", "weight"]]

In [ ]:
# Test bench: good, bad and tricky answers. Which ones fool the rules?
pd.DataFrame([{"case": t["label"], "answer": t["answer"], "score": rewards.score(t["answer"], rules, t["row"])["total"]}
              for t in rewards.TEST_BENCH])

In [ ]:
pd.DataFrame(rewards.score("Don't guarantee anything; check warehouse stock.", rules)["breakdown"])

In [ ]:
# Save + approve a version (RL only accepts approved reward versions)
if SAVE:
    rid = registry.add_reward(rules, note="from notebook"); registry.update("rewards", rid, approved=1); print(rid)
pd.DataFrame(registry.rewards())[["id", "approved", "note"]]